<a href="https://colab.research.google.com/github/studiousdoston/jaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers bitsandbytes datasets peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 79.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.6/996.6 kB 64.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [trl]


In [ ]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, TrainingArguments
import torch
from datasets import load_dataset


In [ ]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# print("Vocab size:", tokenizer.vocab_size)
# print("Special tokens:", tokenizer.special_tokens_map)

# QUANTIZATION
# 4K -> 1080p
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

bnb_config

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16
)

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# Before fine-tuning
# prompt = "Explain what a tokenizer is."
prompt = "A tokenizer is a tool in natural language processing that"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
  output_ids = model.generate(
      **inputs,
      max_new_tokens=80,
      do_sample=True,
      temperature=0.7
  )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))
print(model)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A tokenizer is a tool in natural language processing that separates a sentence or text into smaller units of meaning.

- B Tokenizing involves breaking a sentence or text into smaller units of meaning.

- C Text-to-speech (TTS) conversion converts a sentence or text into audible speech.

- D Speech-to-text conversion converts audible speech into a text.

- E Speech
LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (

In [ ]:
first_block = model.model.layers[0]
print(first_block)
print('\n=============================================\n')
print(first_block.self_attn)

LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
)


LlamaAttention(
  (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
  (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
  (o_proj): Linear

In [ ]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including frozen 4-bit): {total_params:,}")

Total parameters (including frozen 4-bit): 615,606,272


## datasets library | load_dataset
* instruction tuning

In [ ]:
from datasets import load_dataset

dataset = load_dataset("yahma/alpaca-cleaned", split="train")
# dataset[0]
dataset

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json: reconstructing file:   0%|          |  0.00B / 44.3MB            

alpaca_data_cleaned.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 51760
})

In [ ]:
def generate_prompt(example):
  instruction = example["instruction"]
  input_text = example["input"]
  output_text = example["output"]

  if input_text:
    prompt = (
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Input:\n"
        f"{input_text}\n\n"
        "### Response:\n"
        f"{output_text}"
    )
  else:
    prompt = (
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Response:\n"
        f"{output_text}"
    )

  return prompt + tokenizer.eos_token

def formatting_func(example):
  return {"text": generate_prompt(example)}

dataset = dataset.map(formatting_func)


Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [ ]:
dataset[0]["text"]

'### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.</s>'

In [ ]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 51760
})

In [ ]:
# Full Fine-tuning =>
# Cheap Fine-tuning =>
# PEFT  => Parameter Efficient Fine Tuning
# OOM => Out Of Memory


## Setup LoRA config from PEFT

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8, # rank
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

In [ ]:
model = get_peft_model(model, lora_config)

for name, param in model.named_parameters():
  if param.requires_grad:
    param.data = param.data.float()
# keep the trainable LoRA weights in fp32
# the fp16 grad scaler needs this, and fp32 is more stable for training

print({p.dtype for p in model.parameters() if p.requires_grad})

{torch.float32}


In [ ]:
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
# QLoRa
# LoRa

## Setup SFTTrainer from TRL and train the model

In [ ]:
from trl import SFTTrainer, SFTConfig

train_data = dataset.shuffle(seed=42).select(range(5000))

training_args = SFTConfig(
    output_dir="./jaydari_gpt",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    learning_rate=2e-4,
    max_length=512,
    dataset_text_field="text",
    logging_steps=10,
    save_strategy="no",
    bf16=False,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    args=training_args,
)

for name, param in trainer.model.named_parameters():
  if param.requires_grad and param.dtype != torch.float32:
    print("fixing:", name, param.dtype)
    param.data = param.data.float()

trainer.train()

trainer.model.save_pretrained("jaydari_gpt")
tokenizer.save_pretrained("jaydari_gpt")

Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

fixing: base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.bfloat16
fixing: base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight torch.bfloat16
fixing: base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight torch.bfloat16
fixing: base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight torch.bfloat16
fixing: base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight torch.bfloat16
fixing: base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight torch.bfloat16
fixing: base_model.model.model.layers.1.self_attn.v_proj.lora_A.default.weight torch.bfloat16
fixing: base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.weight torch.bfloat16
fixing: base_model.model.model.layers.2.self_attn.q_proj.lora_A.default.weight torch.bfloat16
fixing: base_model.model.model.layers.2.self_attn.q_proj.lora_B.default.weight torch.bfloat16
fixing: base_model.model.model.layers.2.self_attn.v_proj.lor

Step,Training Loss
10,1.411271
20,1.295424
30,1.226333
40,1.340130
50,1.231341
60,1.188297
70,1.195071
80,1.178524
90,1.154352
100,1.208185


('jaydari_gpt/tokenizer_config.json',
 'jaydari_gpt/chat_template.jinja',
 'jaydari_gpt/tokenizer.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer_base = AutoTokenizer.from_pretrained(base_model_id)
model_base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto"
)

model_base.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [ ]:
prompt = "Explain what machine learning is in simple words."

inputs_base = tokenizer_base(prompt, return_tensors="pt").to(model_base.device)

with torch.no_grad():
  output_base = model_base.generate(
      **inputs_base,
      max_new_tokens=120,
      temperature=0.7,
      do_sample=True
  )

print("===== BASE MODEL OUTPUT =====")
print(tokenizer_base.decode(output_base[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== BASE MODEL OUTPUT =====
Explain what machine learning is in simple words. What exactly does it involve, and how does it work?


In [ ]:
!zip -r jaydari_gpt.zip jaydari_gpt

from google.colab import files
files.download("jaydari_gpt.zip")

updating: jaydari_gpt/ (stored 0%)
updating: jaydari_gpt/README.md (deflated 65%)
updating: jaydari_gpt/chat_template.jinja (deflated 60%)
updating: jaydari_gpt/adapter_model.safetensors (deflated 8%)
updating: jaydari_gpt/adapter_config.json (deflated 60%)
updating: jaydari_gpt/tokenizer.json (deflated 85%)
updating: jaydari_gpt/tokenizer_config.json (deflated 46%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Fine-tuned model inference

In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
model_path = "jaydari_gpt"

tokenizer_ft = AutoTokenizer.from_pretrained(model_path)
model_ft = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [ ]:
prompt = """### Instruction:
Explain what machine learning is in simple words.

### Response:
"""
inputs_ft = tokenizer_ft(prompt, return_tensors="pt").to(model_ft.device)

with torch.no_grad():
  output_ft = model_ft.generate(
      **inputs_ft,
      max_new_tokens=120,
      temperature=0.7,
      do_sample=True
  )

print("====== FINE-TUNED MODEL OUPUT ======")
print(tokenizer_ft.decode(output_ft[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


====== FINE-TUNED MODEL OUPUT ======
### Instruction:
Explain what machine learning is in simple words.

### Response:
Machine learning is a branch of artificial intelligence that involves using algorithms to learn from data and make predictions. It can be used in various industries, including healthcare, finance, and marketing, to help companies make better decisions based on the data they have been given. The process of machine learning involves collecting data, creating algorithms, and training them to make predictions. Once the machine learning algorithm is trained, it can be used to make predictions or recommendations based on the data it has learned.


In [ ]:
#---------------
# MODEL TESTING
#---------------


In [ ]:
!unzip jaydari_gpt.zip

Archive:  jaydari_gpt.zip
   creating: jaydari_gpt/
  inflating: jaydari_gpt/README.md   
   creating: jaydari_gpt/checkpoint-3500/
  inflating: jaydari_gpt/checkpoint-3500/README.md  
  inflating: jaydari_gpt/checkpoint-3500/trainer_state.json  
  inflating: jaydari_gpt/checkpoint-3500/scheduler.pt  
  inflating: jaydari_gpt/checkpoint-3500/special_tokens_map.json  
  inflating: jaydari_gpt/checkpoint-3500/tokenizer.model  
  inflating: jaydari_gpt/checkpoint-3500/rng_state.pth  
  inflating: jaydari_gpt/checkpoint-3500/adapter_config.json  
  inflating: jaydari_gpt/checkpoint-3500/tokenizer.json  
  inflating: jaydari_gpt/checkpoint-3500/chat_template.jinja  
  inflating: jaydari_gpt/checkpoint-3500/optimizer.pt  
  inflating: jaydari_gpt/checkpoint-3500/adapter_model.safetensors  
  inflating: jaydari_gpt/checkpoint-3500/training_args.bin  
  inflating: jaydari_gpt/checkpoint-3500/tokenizer_config.json  
   creating: jaydari_gpt/checkpoint-1750/
  inflating: jaydari_gpt/checkpoint-1

In [ ]:
!pip install -q langdetect

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig
import torch

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_dir = "/content/jaydari_gpt"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
tokenizer.pad_token = tokenizer.eos_token

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# Load fine-tuned adapter
model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

In [ ]:
from langdetect import detect_langs

def ask_jaydari_gpt(prompt:str, max_new_tokens=256):
  try:
    langs = detect_langs(prompt)
    lang_code = langs[0].lang
    confidence = langs[0].prob
    print(f"🌏 Detected lang: {lang_code} ({confidence:.2f})")
  except:
    lang_code = "unknown"
    confidence = 0.0

  # Block only if we are confident it's NOT english
  if lang_code != "en" and confidence >= 0.95 and len(prompt) > 20:
    return "Sorry, you should ask a question only in English."

  # Proceed with generation
  full_prompt = f"### Instruction:\n{prompt.strip()}\n\n### Response:"
  inputs = tokenizer(full_prompt, return_tensors="pt").to("cuda")

  outputs = model.generate(
      **inputs,
      max_new_tokens=max_new_tokens,
      temperature=0.7,
      top_p=0.9,
      do_sample=True,
      eos_token_id=tokenizer.eos_token_id
  )

  decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
  return decoded[len(full_prompt):].strip()

In [ ]:
ask_jaydari_gpt("Tell me about Python")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🌏 Detected lang: en (1.00)


'Python is a high-level, interpreted, object-oriented programming language that is widely used for web development, data analysis, and machine learning. It is designed to be easy to learn and use, with a focus on simplicity, readability, and efficiency. Python is known for its flexibility, readability, and ease of use, making it a popular choice for developers. Its use in web development allows for efficient and effective programming, as it enables developers to create and manipulate data with ease. Python is also used in data analysis, and machine learning, with tools such as NumPy, SciPy, and Pandas being popular for data manipulation and analysis. Python is an open-source language, which means that it is freely available and can be downloaded and used for personal and commercial projects. Its popularity has made it a popular choice for web development, data analysis, and machine learning.'